In [ ]:
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch import optim

# 1. Imports from Repo
repo_dir = "/content/stk-mat2011"
sys.path.append(f"{repo_dir}/code/scripts")
from nnhmm import NNHMM
from p_drive import mount_drive

mount_drive()
data_path = "/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega/eurusd_mega_master_21_26.parquet"

# Load & Convert to Torch
df = pd.read_parquet(data_path)
y = torch.tensor(df['returns'].values, dtype=torch.float32).unsqueeze(1)

# Device Auto-Detect
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device} | Sequence Length: {len(y)}")

In [ ]:
# Initialize Heavyweight Model
model = NNHMM(n_states=2, input_dim=1).to(device)
y = y.to(device)

# Professional Optimizer (Adam with Learning Rate Scheduler)
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)

EPOCHS = 1000 # Start with 1k while on CPU, move to 50k for GPU
losses = []

print("🚀 Starting SGD-HMM Training...")

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    
    # Compute Negative Log Likelihood (NLL)
    log_lik = model(y)
    loss = -log_lik / len(y) # Per-sample loss
    
    # Backprop exactly like your PINN files
    loss.backward()
    optimizer.step()
    scheduler.step(loss)
    
    losses.append(loss.item())
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:04d} | NLL Loss: {loss.item():.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

print("🏁 Training complete.")

In [ ]:
# Extract learned Regimes
with torch.no_grad():
    trans = model.transition_matrix.cpu().numpy()
    means = model.means.cpu().numpy().flatten()
    stds = torch.exp(model.log_std).cpu().numpy().flatten()

print("-" * 50)
print("LEARNED 5-YEAR REGIMES:")
print("-" * 50)
for i in range(2):
    print(f"Regime {i}: Mean={means[i]:.8f}, Std={stds[i]:.8f}")

print("\nTRANSITION MATRIX:")
print(trans)

plt.plot(losses)
plt.title("Neural HMM Convergence (NLL)")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.show()